# Lesson 4 — Gradient flow: watching backprop and diagnosing training

When a network fails to train, the code usually runs without errors — the loss just won't go down. To debug that you need to
**look inside**: at the graph backprop walks, at the activations each layer produces, and at the gradients that reach each layer.

**You will learn to**
1. Watch `backward()` run one node at a time
2. Count the cost of scalar autograd — and see why real frameworks use tensors
3. Diagnose **exploding / vanishing activations** and fix them with proper initialization
4. Detect **dead ReLUs** caused by a too-large learning rate
5. Watch a decision boundary form during training

Part 1 draws graphs with graphviz (`brew install graphviz` / `apt install graphviz`, then `pip install graphviz`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so `import nanograd` works from the lessons/ folder without installing

def check(name, test):
    """ run a test function; print a tick, a cross, or a reminder if not implemented yet """
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet")
        return
    except Exception as e:
        print(f"❌ {name}: raised {type(e).__name__}: {e}")
        return
    print(f"{'✅' if ok else '❌'} {name}")

In [ ]:
import math, random, statistics
import matplotlib.pyplot as plt
from IPython.display import display
from nanograd.engine import Value
from nanograd.nn import MLP
from nanograd import diagnostics as dg
from nanograd.viz import draw_dot, plot_decision_boundary

## Part 1 — Backprop, one step at a time

`backward()` sorts the graph topologically and then calls each node's `_backward` from the output back to the inputs.
`nanograd.diagnostics.backward_steps` does the same thing, but pauses after every node so we can look.
In the pictures, **orange** is the node whose `_backward` just ran (it pushed its gradient into its children) and **green** nodes
are finished.

In [ ]:
a, b, c, f = Value(2.0), Value(-3.0), Value(10.0), Value(-2.0)
e = a * b
d = e + c
L = d * f
print("forward: L =", L.data)

### Exercise 1.1 — backprop by hand
Before watching, compute $\partial L/\partial x$ for each leaf by hand with the chain rule. (Write numbers, not code that calls `backward`.)

In [ ]:
my_grads = {"a": None, "b": None, "c": None, "f": None}  # YOUR ANSWERS HERE

In [ ]:
def test_hand_grads():
    if any(v is None for v in my_grads.values()):
        raise NotImplementedError
    a2, b2, c2, f2 = Value(2.0), Value(-3.0), Value(10.0), Value(-2.0)
    ((a2 * b2 + c2) * f2).backward()
    true = {"a": a2.grad, "b": b2.grad, "c": c2.grad, "f": f2.grad}
    for k in true:
        print(f"  dL/d{k}: yours {my_grads[k]:>5}   true {true[k]:>5}")
    return all(abs(my_grads[k] - true[k]) < 1e-9 for k in true)

check("hand-computed gradients", test_hand_grads)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
# L = d * f      -> dL/df = d = (a*b + c) = 4,   dL/dd = f = -2
# d = e + c      -> dL/dc = dL/dd * 1 = -2,      dL/de = -2
# e = a * b      -> dL/da = dL/de * b = 6,       dL/db = dL/de * a = -4
my_grads = {"a": 6.0, "b": -4.0, "c": -2.0, "f": 4.0}
check("hand-computed gradients", test_hand_grads)

In [ ]:
done = set()
for i, (node, order) in enumerate(dg.backward_steps(L)):
    print(f"step {i}: ran _backward of node with data {node.data} (op '{node._op or 'leaf'}')")
    display(draw_dot(L, highlight=node, done=done))
    done.add(node)

Notice:
* The output's gradient starts at 1 ($\partial L / \partial L$).
* A node's gradient is **complete** before its `_backward` runs — that's what the topological order guarantees.
* Leaves (`a`, `b`, `c`, `f`) have nothing to push further; their `_backward` is a no-op.

## Part 2 — How big is the graph?

nanograd works on individual scalars. Let's count how many `Value` objects one forward pass creates.

In [ ]:
random.seed(0)
model = MLP(2, [16, 16, 1])
out = model([Value(0.5), Value(-1.0)])
print("parameters:            ", len(model.parameters()))
print("graph nodes, 1 example:", dg.count_nodes(out))

### Exercise 2.1 — count the multiplications
For an `MLP(nin, sizes)`, each layer with `n_in` inputs and `n_out` neurons performs `n_in * n_out` multiplications
(one per weight). Write `count_params(nin, sizes)` (weights **plus** biases) and `count_mults(nin, sizes)`.

In [ ]:
def count_params(nin, sizes):
    # YOUR CODE HERE
    raise NotImplementedError

def count_mults(nin, sizes):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_counts():
    ok = True
    for nin, sizes in [(2, [16, 16, 1]), (3, [4, 4, 1]), (784, [128, 10])]:
        random.seed(0)
        ok &= count_params(nin, sizes) == len(MLP(nin, sizes).parameters())
    return ok and count_mults(2, [16, 16, 1]) == 2*16 + 16*16 + 16*1

check("count_params and count_mults", test_counts)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def count_params(nin, sizes):
    dims = [nin] + sizes
    return sum(dims[i] * dims[i+1] + dims[i+1] for i in range(len(sizes)))

def count_mults(nin, sizes):
    dims = [nin] + sizes
    return sum(dims[i] * dims[i+1] for i in range(len(sizes)))

check("count_params and count_mults", test_counts)

In [ ]:
# a small image classifier (28x28 pixels -> 10 digits) in nanograd terms
nin, sizes = 784, [128, 10]
per_example = count_mults(nin, sizes)
print(f"MLP(784, [128, 10]): {count_params(nin, sizes):,} parameters, {per_example:,} multiplications per example")
print(f"one batch of 32 examples: ~{32 * per_example * 2:,} Value objects (a multiply and an add per weight)")

Each of those `Value`s is a Python object with a closure. A modern GPU does a 32×784×128 matrix multiply as **one** operation.
That's the leap to *tensors*: the same chain rule, applied to whole arrays at once. Conceptually nothing changes — the backward of
`Y = X @ W` is `dX = dY @ W.T` and `dW = X.T @ dY`, which is exactly what nanograd computes one scalar at a time.

## Part 3 — Initialization: exploding and vanishing activations

Consider a deep network: 6 hidden layers of 16 ReLU neurons. If each layer scales its input's spread by a factor `g`, after 6 layers
the spread is $g^6$ — tiny if $g < 1$, enormous if $g > 1$. Let's measure the standard deviation of each layer's activations
on the moons inputs, for three ways of initializing the weights:

* `'uniform'` — the original scheme here, $w \sim U(-1, 1)$, regardless of layer width
* `0.1` — small Gaussian weights, std 0.1
* `'he'` — Gaussian with std $\sqrt{2 / n_{in}}$

In [ ]:
from nanograd.datasets import make_moons
from nanograd.losses import hinge

X, y = make_moons(n_samples=50, noise=0.1, seed=0)
y_pm = [2 * t - 1 for t in y]

def inspect(init, act='relu', depth=6, width=16, seed=0):
    """ activation stds and mean |weight grad| for every layer of a deep MLP """
    random.seed(seed)
    model = MLP(2, [width] * depth + [1], act=act, init=init)
    acts = dg.layer_activations(model, X)[:-1]          # hidden layers only
    loss = sum(hinge(model([Value(xi) for xi in x]), t) for x, t in zip(X, y_pm)) * (1.0 / len(X))
    model.zero_grad(); loss.backward()
    stds = [statistics.pstdev([a for sample in layer for a in sample]) for layer in acts]
    grads = [statistics.mean(abs(g) for g in layer) for layer in dg.layer_grads(model)]
    return model, acts, stds, grads

def plot_inspection(inits, act='relu'):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    for init in inits:
        model, acts, stds, grads = inspect(init, act)
        axes[0].plot(range(1, len(stds) + 1), stds, 'o-', label=f"init={init}")
        axes[1].plot(range(1, len(grads) + 1), grads, 'o-', label=f"init={init}")
        last = [a for sample in acts[-1] for a in sample]
        axes[2].hist(last, bins=40, alpha=0.5, label=f"init={init}")
    axes[0].set_yscale('log'); axes[0].set_xlabel('hidden layer'); axes[0].set_title('activation std per layer')
    axes[1].set_yscale('log'); axes[1].set_xlabel('layer'); axes[1].set_title('mean |weight gradient| per layer')
    axes[2].set_yscale('log'); axes[2].set_title('activations of the last hidden layer')
    for ax in axes: ax.legend()
    plt.tight_layout(); plt.show()

plot_inspection(['uniform', 0.1, 'he'])

* With **`uniform`**, the spread grows layer after layer (*exploding* activations): a later layer sees inputs several times larger
  than an earlier one, and the gradients are correspondingly large and uneven.
* With **std 0.1**, the signal shrinks by ~3× per layer and has all but vanished by layer 6 (*vanishing*); the gradients are
  ~1,000× smaller than with He init, so the network barely learns.
* With **He** init, the spread stays roughly constant through every layer.

### Exercise 3.1 — derive He initialization
For one neuron $z = \sum_{i=1}^{n} w_i x_i$ with independent zero-mean weights, $\text{Var}(z) = n \cdot \text{Var}(w) \cdot E[x^2]$.
A ReLU zeroes the negative half of its input, so $E[\text{relu}(z)^2] = \tfrac12 \text{Var}(z)$.

Find the weight standard deviation `my_std(n)` that keeps $E[x^2]$ the same from one layer to the next. Then the cell below builds
a network with your std (passing a float as `init` means "Gaussian with this std") and checks that the activations neither
explode nor vanish.

In [ ]:
def my_std(n):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_my_std():
    stds = inspect(my_std(16))[2]
    print("   activation std per layer:", " ".join(f"{s:.2f}" for s in stds))
    return 0.4 < stds[-1] / stds[0] < 2.5

check("my_std keeps activations stable", test_my_std)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def my_std(n):
    # we need n * Var(w) * 1/2 = 1  ->  Var(w) = 2/n  ->  std = sqrt(2/n)   (He et al., 2015)
    return math.sqrt(2 / n)

check("my_std keeps activations stable", test_my_std)

For **tanh** there's no "zeroing half" factor, so the matching choice is std $= \sqrt{1/n}$ (`init='xavier'`). With weights that are
too large, tanh doesn't explode — it **saturates** at ±1, where its slope $1 - \tanh^2$ is ~0, so gradients vanish instead:

In [ ]:
plot_inspection([1.0, 'xavier'], act='tanh')
for init in (1.0, 'xavier'):
    _, acts, _, _ = inspect(init, act='tanh')
    print(f"init={init!s:<7} fraction of saturated tanh units (|a| > 0.99) per layer:",
          " ".join(f"{dg.saturated_fraction(l):.2f}" for l in acts))

## Part 4 — Dead ReLUs

A ReLU neuron whose input is negative for **every** example outputs 0 and passes back a gradient of 0. Its weights never change
again: the neuron is *dead*. A too-large learning rate is the classic cause — one big update can push a neuron's weights so that it
never activates again.

### Exercise 4.1 — measure dead neurons
`dg.layer_activations(model, X)` returns `acts[layer][sample][neuron]`. Write `dead_fraction(acts_layer)`: the fraction of neurons
in a layer that output exactly 0 for every sample.

In [ ]:
def dead_fraction(acts_layer):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check("dead_fraction small example", lambda: dead_fraction([[0.0, 1.0, 0.0], [0.0, 0.0, 2.0]]) == 1 / 3)
check("dead_fraction matches library", lambda: all(
    dead_fraction(l) == dg.dead_fraction(l) for l in inspect('uniform')[1]))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def dead_fraction(acts_layer):
    n_neurons = len(acts_layer[0])
    dead = sum(1 for j in range(n_neurons) if all(sample[j] == 0 for sample in acts_layer))
    return dead / n_neurons

check("dead_fraction small example", lambda: dead_fraction([[0.0, 1.0, 0.0], [0.0, 0.0, 2.0]]) == 1 / 3)
check("dead_fraction matches library", lambda: all(
    dead_fraction(l) == dg.dead_fraction(l) for l in inspect('uniform')[1]))

In [ ]:
from nanograd import optim

def train(lr, steps=30, seed=0):
    random.seed(seed)
    model = MLP(2, [16, 16, 1])
    opt = optim.SGD(model.parameters(), lr=lr)
    dead, losses = [], []
    for k in range(steps + 1):
        dead.append([dead_fraction(l) for l in dg.layer_activations(model, X)[:-1]])
        if k == steps:
            break
        loss = sum(hinge(model([Value(xi) for xi in x]), t) for x, t in zip(X, y_pm)) * (1.0 / len(X))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.data)
    return model, losses, dead

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for lr in (0.1, 2.0):
    model, losses, dead = train(lr)
    axes[0].plot(losses, label=f"lr={lr}")
    axes[1].plot([sum(d) / len(d) for d in dead], label=f"lr={lr}")
    print(f"lr={lr}: dead fraction per hidden layer before {dead[0]} -> after {dead[-1]}, final loss {losses[-1]:.3f}")
axes[0].set_title('loss'); axes[0].legend(); axes[1].set_title('fraction of dead ReLUs'); axes[1].legend()
plt.show()

With `lr=2.0` over half of the hidden neurons die within a few steps, and that capacity is gone for good — even if we lowered the
learning rate afterwards, dead neurons receive no gradient to recover. Remedies used in practice: smaller learning rates or warm-up,
Adam, careful initialization, or activations that never have exactly zero slope (Leaky ReLU, GELU).

### Exercise 4.2 — Leaky ReLU (stretch goal)
Add a `leaky_relu(v, alpha=0.01)` op (output $x$ for $x > 0$, $\alpha x$ otherwise), gradient-check it, and argue in one sentence why
a Leaky ReLU neuron can never be permanently dead.

In [ ]:
# YOUR CODE HERE

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
from nanograd.gradcheck import gradcheck

def leaky_relu(v, alpha=0.01):
    out = Value(v.data if v.data > 0 else alpha * v.data, (v,), 'leaky_relu')
    def _backward():
        v.grad += (1.0 if v.data > 0 else alpha) * out.grad
    out._backward = _backward
    return out

print("gradcheck at x=+0.7:", gradcheck(lambda v: leaky_relu(v[0]), [0.7]))
print("gradcheck at x=-0.7:", gradcheck(lambda v: leaky_relu(v[0]), [-0.7]))
# Its slope is never exactly 0 (it is alpha for negative inputs), so some gradient always reaches the
# weights, and a neuron pushed into the negative region can still be pulled back out.

## Part 5 — Watching a decision boundary form

Finally, let's watch learning happen. We train with Adam and snapshot the model at a few steps.

In [ ]:
import copy

random.seed(1337)
model = MLP(2, [16, 16, 1])
opt = optim.Adam(model.parameters(), lr=0.05)
snap_at = [0, 3, 8, 15, 30, 60]
snapshots = {}
for k in range(max(snap_at) + 1):
    if k in snap_at:
        snapshots[k] = copy.deepcopy(model)
    loss = sum(hinge(model([Value(xi) for xi in x]), t) for x, t in zip(X, y_pm)) * (1.0 / len(X))
    opt.zero_grad(); loss.backward(); opt.step()
    if k in snap_at:
        snapshots[k].loss = loss.data

fig, axes = plt.subplots(1, len(snap_at), figsize=(3 * len(snap_at), 3))
for ax, k in zip(axes, snap_at):
    plot_decision_boundary(snapshots[k], X, y, ax=ax, h=0.08, title=f"step {k}, loss {snapshots[k].loss:.2f}")
plt.tight_layout(); plt.show()

Early on the boundary is a rough straight-ish cut; the network first learns the coarse split, then bends the boundary to wrap around
each moon. Watching this is one of the best ways to build intuition for what "the loss going down" actually means.

## Summary — a debugging checklist
| symptom | look at | likely cause | fix |
|---|---|---|---|
| loss stuck from the start | activation std per layer | vanishing signal (weights too small, saturated tanh) | Xavier / He init |
| loss is NaN or huge | activation std, gradient size | exploding signal or lr too big | proper init, smaller lr |
| loss plateaus after a few big steps | fraction of dead ReLUs | lr too large | smaller lr, Adam, Leaky ReLU |
| gradients wrong | `gradcheck` (Lesson 1) | buggy `_backward` | fix the op |